# تدريب نموذج كود خاص بيك — Qwen2.5-Coder-1.5B + LoRA
# Fine-tune your own code model — الإصدار النظيف v2

> 🆕 **دي النسخة المعاد بناؤها من الصفر (v2).** لو النوتبوك القديمة مفتوحة عندك،
> **اقفلها وافتح دي من جديد** — النسخة دي مثبتة فيها إصدارات المكتبات بالظبط
> (Exact pins) ومتحقق من كل إعداد فيها من كود المكتبات نفسه، عشان غلطتين
> وقعنا فيهم قبل كده ميحصلوش تاني:
> 1. `max-seq-length` اتشالت من TRL الجديدة → البديل الصح `max_length` ✅
> 2. خطأ `BFloat16` في التدريب → سببه اتفهم واتصلح من جذره (اشرحه تحت عند خلية التدريب) ✅

النوتبوك دي بتعمل **Fine-tune** (تعليم إضافي) لنموذج مفتوح جاهز هو
`Qwen/Qwen2.5-Coder-1.5B-Instruct` — النموذج ده أصلاً بيفهم لغات برمجة كتير
لأنه اتدرب على كود بكميات ضخمة. إحنا **مش** بندرّب من الصفر؛ إحنا بنعلّمه
أسلوب ونوع مهام معين باستخدام تقنية **LoRA / QLoRA** على داتاست تعليمات كود
حقيقية (`ise-uiuc/Magicoder-OSS-Instruct-75K`)، والنتيجة بتترفع على حسابك
في Hugging Face: **darklink56/qwen-coder-1.5b-lora**.

**Requirements / المتطلبات:**
- Runtime must be a **free GPU (T4)**: `Runtime → Change runtime type → T4 GPU`
- A Hugging Face **Write** token (هتحطه في خلية الدخول رقم 3 — الشرح هناك)
- Cost: **0** — everything runs on free tiers (Colab free + HF free)
- اسم المستخدم مثبّت بالفعل على `darklink56` في خلية الإعدادات (Config)


In [ ]:
# 1) Check the GPU — لازم يطلع لك كارت T4 (أو غيره من كروت Colab المجانية)
!nvidia-smi

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU, then run again.")


In [ ]:
# 2) Install dependencies — إصدارات مثبتة بالظبط (Exact pins) ومتحقق من توافقها
# v2: مفيش نطاقات مفتوحة هنا — كل مكتبة برقم إصدار محدد، وTRL 0.29.1 نفسها
# بتشترط transformers>=4.56.2 و accelerate>=1.4.0 و datasets>=3.0.0 و peft>=0.8.0
# والإصدارات دي جوّه الشروط دي بالظبط.
!pip install -q "trl==0.29.1" "transformers==4.57.6" "peft==0.22.0" "accelerate==1.16.0" "bitsandbytes==0.50.2" "datasets==4.8.5"

import trl, transformers, peft, accelerate, datasets
print("trl", trl.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| accelerate", accelerate.__version__,
      "| datasets", datasets.__version__)

# Self-check: لو أي إصدار اختلف عن المثبت، وقّف هنا ومتكملش — كده البيئة مختلفة
assert trl.__version__ == "0.29.1", f"trl version mismatch: {trl.__version__}"
assert transformers.__version__ == "4.57.6", f"transformers version mismatch: {transformers.__version__}"
print("✅ Versions OK — البيئة مطابقة للنسخة المتحقق منها")


In [ ]:
# 3) Log in to Hugging Face — التوكن سر، ومش مكتوب في النوتبوك أبداً
#
# الطريقة الأساسية: التوكن بييجي من Colab Secrets (أيقونة 🔑 على الشمال →
# سر اسمه HF_TOKEN بالظبط + تفعيل "Notebook access").
#
# 📱 لو مش لاقي أيقونة المفتاح (بيحصل على الموبايل): ولا يهمك — لو السر مش
# موجود، الخلية دي هتفتح لك **خانة توكن تحت الخلية نفسها**؛ الصق فيها توكن
# الـ Write بتاعك ودوس Enter، وهيكمل عادي. التوكن بيتحط هناك بس،
# و**متبعتهوش في الشات أبداً**.
import os

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

from huggingface_hub import login
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets (HF_TOKEN)")
else:
    # Fallback: interactive login — هيظهر لك خانة Token تحت الخلية، الصق التوكن فيها
    print("ℹ️ مفيش HF_TOKEN في الأسرار — هيظهر لك خانة توكن دلوقتي، الصق توكن الـ Write فيها:")
    login()
    print("✅ Logged in interactively")


In [ ]:
# 4) Config — الإعدادات (اسم المستخدم مثبّت على darklink56، مفيش تعديل مطلوب)
BASE_MODEL   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # النموذج الأساسي (Apache-2.0)
HF_USERNAME  = "darklink56"                          # اسم مستخدمك على Hugging Face
OUTPUT_REPO  = f"{HF_USERNAME}/qwen-coder-1.5b-lora" # الأدابتور هيترفع هنا

DATASET_NAME = "ise-uiuc/Magicoder-OSS-Instruct-75K" # أعمدة: problem / solution
SAMPLE_SIZE  = 3000        # أول تشغيلة: 3000 مثال بس عشان تخلص في جلسة مجانية
NUM_EPOCHS   = 1
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH = 1       # مناسب لذاكرة T4 (16GB) مع تحميل 4-bit
GRAD_ACCUM = 16             # 2 × 8 = effective batch 16
MAX_SEQ_LEN = 768

print("Base model :", BASE_MODEL)
print("Output repo:", OUTPUT_REPO)
print("Dataset    :", DATASET_NAME, "| samples:", SAMPLE_SIZE)


In [ ]:
# 5) Load the dataset and format it with the model's chat template
# الداتاست أعمدتها problem (المطلوب) و solution (الحل) — بنحوّل كل صف لمحادثة:
# user = problem  →  assistant = solution
# ملحوظة: إحنا مش بنعمل توكننة بنفسنا — SFTTrainer في TRL 0.29.1 بيتوكن عمود
# "text" بنفسه (ده سلوكه الموثق في كوده)، فإحنا بنجهز النص بس.
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
# Qwen معندهاش pad token منفصل — بنثبته على <|endoftext|> (نفس اللي النموذج بيستخدمه)
tokenizer.pad_token = "<|endoftext|>"

raw = load_dataset(DATASET_NAME, split=f"train[:{SAMPLE_SIZE}]")
print(raw)

def to_text(example):
    messages = [
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]},
    ]
    # apply_chat_template بيطلع النص بنفس صيغة Qwen (ChatML) + توكن النهاية
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = raw.map(to_text, remove_columns=raw.column_names)
print(train_ds[0]["text"][:600])


## 6) Training — التدريب (اقرأ دي قبل ما تشغّل الخلية الجاية)

- تحميل النموذج بـ **4-bit (QLoRA)** وبصيغة **float16** صراحةً
  (`torch_dtype=torch.float16`) عشان يدخل في ذاكرة T4.
- **ليه `fp16=False` و `bf16=False` في إعدادات التدريب؟** ده إصلاح الغلطة
  التانية من جذورها: مكتبة TRL 0.29.1 نفسها بتحوّل أوزان أدابتور QLoRA
  لصيغة **bf16** أثناء التدريب (دي توصية ورقة QLoRA الأصلية). لو فعّلنا
  `fp16=True`، المدرّب بيستخدم أداة اسمها GradScaler مش بتدعم bf16 على
  كروت T4 فبيقع بخطأ `NotImplementedError ... BFloat16`. الحل: نقفل
  الـ Mixed Precision من المدرّب خالص (مفيش Scaler أصلاً)، والحساب نفسه
  بيكمل بـ float16 جوه bitsandbytes. كده الخطأ مستحيل يحصل.
- **LoRA**: r=16, alpha=32, على كل طبقات الـ projections
  (q/k/v/o + gate/up/down) — بنعلّم أدابتور صغير بس، مش النموذج كله.
- `gradient_checkpointing_kwargs={"use_reentrant": False}` +
  `model.enable_input_require_grads()` — بيمنعوا غلطة كلاسيكية اسمها
  "element 0 of tensors does not require grad".
- أول تشغيلة بـ 3000 مثال و epoch واحدة: توقّع وقت في حدود جلسة Colab
  مجانية واحدة تقريباً (بيختلف حسب طول الأمثلة وزحمة Colab).


In [ ]:
# 6) Train with TRL SFTTrainer + LoRA (QLoRA 4-bit) — v2, verified API (TRL 0.29.1)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,  # صريح: النموذج كله fp16، مش bf16 الافتراضي
    device_map="auto",
)
model.config.use_cache = False  # لازم مع gradient checkpointing
model.enable_input_require_grads()  # لازم مع gradient checkpointing + LoRA

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="qwen-coder-1.5b-lora-out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",            # هنرفع الأدابتور في الآخر بس
    fp16=False,                    # ⚠️ لازم False — شوف شرح الغلطة التانية فوق
    bf16=False,                    # ⚠️ لازم False — T4 مش بيدعم bf16 أصلاً
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    max_length=MAX_SEQ_LEN,        # TRL الجديدة: max_length (مفيش max-seq-length خلاص)
    packing=False,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
print("✅ Training finished")


In [ ]:
# 7) Quick test — جرّب النموذج بعد التدريب على 3 لغات مختلفة
import torch

test_prompts = [
    "Write a Python function that returns the nth Fibonacci number using memoization.",
    "Write a JavaScript function that debounces another function by 300ms.",
    "Write a SQL query that returns the top 3 customers by total order amount from tables customers and orders.",
]

model.config.use_cache = True
model.eval()

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("=" * 80)
    print("PROMPT:", prompt)
    print("-" * 80)
    print(answer)


In [ ]:
# 8) Push the adapter + tokenizer to YOUR Hugging Face account
# النتيجة أدابتور LoRA صغير (مش النموذج كله) — بيتحمّل فوق النموذج الأساسي وقت التشغيل
trainer.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"✅ Adapter pushed to: https://huggingface.co/{OUTPUT_REPO}")


In [ ]:
# 9) (Optional) Save a backup zip to Google Drive — اختياري
# شيل علامات التعليق (#) من السطور الجاية لو عايز نسخة احتياطية على Drive:
#
# from google.colab import drive
# drive.mount("/content/drive")
# !cd qwen-coder-1.5b-lora-out && zip -r /content/drive/MyDrive/qwen-coder-1.5b-lora.zip checkpoint-* adapter_model.safetensors adapter_config.json 2>/dev/null || zip -r /content/drive/MyDrive/qwen-coder-1.5b-lora.zip .
# print("Saved to Drive")
print("Skipped — optional cell. The adapter is already on Hugging Face after cell 8.")


## Troubleshooting — لو حاجة وقفت

| المشكلة | الحل |
|---|---|
| **Versions mismatch في خلية التثبيت (2)** | البيئة مختلفة عن المثبتة. اعمل `Runtime → Restart session` وشغّل من الأول — التثبيت هيظبط الإصدارات بالظبط. |
| **CUDA out of memory (OOM)** | قلّل `PER_DEVICE_BATCH` لـ 1 وزوّد `GRAD_ACCUM` لـ 16، أو قلّل `MAX_SEQ_LEN` لـ 768، أو قلّل `SAMPLE_SIZE`. |
| **Runtime disconnected / session ended** | Colab المجاني بيفصل بعد فترة خمول أو طول جلسة. الأدابتور بيضيع لو الفصل حصل قبل خلية الرفع (8) — عشان كده أول تشغيلة صغيرة (3000 مثال). للتشغيلات الأكبر: قسّم الداتا على كذا جلسة، أو فعّل حفظ checkpoint كل فترة وارفع كل مرحلة. |
| **401 / 403 من Hugging Face** | اتأكد إن التوكن من نوع **Write**. لو دخلت من خانة التوكن التفاعلية (خلية 3) ولسه في خطأ، اعمل توكن Write جديد وجرّب تاني. |
| **Repository not found عند الرفع** | ده طبيعي لو الريبو مش موجود — `push_to_hub` بينشئه تلقائياً طالما التوكن Write واسم المستخدم صح. |
| **النموذج بيرد رد ضعيف بعد التدريب** | متوقّع جزئياً مع 3000 مثال و epoch واحدة — دي تشغيلة إثبات فكرة. زوّد `SAMPLE_SIZE` و `NUM_EPOCHS` لـ 2 في التشغيلات الجاية وهتلاحظ الفرق. |

الغلطتين بتوع النسخة الأولى (v1) **مش هيظهروا هنا**: `max-seq-length`
اتبدلت بـ `max_length`، وخطأ `BFloat16` اتصلح بقفل `fp16/bf16` في إعدادات
التدريب مع تثبيت تحميل النموذج على `float16` (الشرح الكامل فوق خلية التدريب).

## Next steps — اللي بعده
1. جرّب الأدابتور محلياً بـ `inference.py` اللي في الريبو.
2. زوّد حجم الداتا تدريجياً (10K ثم أكتر) في جلسات منفصلة.
3. ادمج الأدابتور في النموذج (merge) أو صدّره GGUF عشان يشتغل محلياً بأدوات زي Ollama / llama.cpp — شوف قسم Roadmap في README.
